# Environment Setup

In [1]:
import getpass
cur_user = getpass.getuser()

In [2]:
%cd /home/{cur_user}/code/algodev/

/home/orr/code/algodev


In [3]:
import inu.ds
from pathlib import Path

test_path = Path(inu.ds.__file__).parent / 'tests'
%cd {test_path}

/home/orr/.conda/envs/dev39/lib/python3.9/site-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.16.5 and <1.23.0 is required for this version of SciPy (detected version 1.23.1
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


/home/orr/code/algodev/inu/ds/tests


In [4]:

%pylab inline
from inu.ds import *
from inu.ds.transform import apply_transforms
from inu.visual.insight import *
import pandas as pd

# FORMATTING SETTINFS
from qbstyles import mpl_style
mpl_style(dark=True)

pd.set_option('max_colwidth', 15)

np.set_string_function(array_str)   # show only arrays info 

%pylab is deprecated, use %matplotlib inline and import the required libraries.
Populating the interactive namespace from numpy and matplotlib


# Datasets and `PathSchemes`
Structures of datasets and their formal definitions using `PathScheme` are described elsewhere  
(see, `inu.ds` docs and `tests/ds_user_manual.ipynb` notebook).

Here we just remind that `inu.ds` package considers:
  - data set as a collection of _data items_, 
  - each item as collection of labels `{category: value}`
  
### Specific Labeling Conventions:
Those are default for the framework (can be changed):
 - `path` - location of the items data 
 - `data` - when loaded as an object into memory
 - `transforms` - used by scheme to define transforms required to "normalize" the data
 - `transformed` - desribes how data _has been_ transformed from the "normal" state
 
Those are more domain specific:

 - `alg`  - category is a _method_ used to produce the data.   
 - `kind` - provides additional specifics about its nature

## From `PathScheme`s to `DataCollection`
Processing of experiments may require coordinated acces to _multiple_ related datasets.  
In this example we have two datasets:
   1. The original stereo dataset (Middlebury) with _left, right_ images and _disparity_ Ground Truth
   2. Disparities calculated from the Middlebury stereo pairs by different algorithms 

Analysis of those results require coordinated access to both data sets, when, for example,  
for every _scene_ the ground truth (GT) and disparities of evaluated algorithms must be compaired.

`DataCollection` class combines those datasets, to allow flexible queries, iterations,  
arbitrary data transformations, and other data logistics capabilities.

In this case schemes are stored in the respective folder of every dataset,   
so we can initialize the schemes by those folders:

In [5]:
datasets  = ['/mnt/t/DataSets/depth/Middlebury/2014_small', '/mnt/t/DataSets/evals/Middlebury']
schemes = [*map(PathScheme.from_yaml, datasets)]

The structure of the datsets can be examined, to see which `{categories}` are defined,  
and if there are some common ones:

In [6]:
print('\n'.join(map(lambda s: f"{s} defines: {s.categories}", schemes)))
s1, s2 = schemes
print("Common categories:", f"{s1.categories.intersection(s2.categories)}")

Scheme '…': [chn,view,alg,scene,kind,transforms,ext] defines: {'chn', 'view', 'alg', 'scene', 'kind', 'transforms', 'ext'}
Scheme '…': [ext,transforms,view,alg,scene,kind,transformed,recode] defines: {'ext', 'transforms', 'view', 'alg', 'scene', 'kind', 'transformed', 'recode'}
Common categories: {'transforms', 'ext', 'view', 'alg', 'scene', 'kind'}


## Creating DataCollection

### Data Structure
`DataCollection` gathers elements from all the datasets as a single table (`pandas.DataFrame`) with  
 - categories as __columns__, and
 - data items as __rows__

Logically the categories can be separated into the _index_ and _data_ parts.  

 - __index__ uses labels more suitable for querying, while
 - __data__ is usually meant for processing   

though the structure is flexible, can be changed dynamically, and even some categories could be assigned to both.  
Technically, the multi-category index is implemented as `pandas.MultiIndex` for the rows.

By default all categories are set as _index_ except the `path`, `transfomed`, `transforms` and `data`.


### Cleanups
Meaningful merge of datasets requires all the common categories to share sematical meaning.  
Those, and also categories not needed for specific work should be **dropped** .   
Basic cleanup may also include filtering out certain rows (unneeded data elements).  
In this case, that would be certain masks `alg==nocc` provided by the Middlebury datasets.

You can filter explicitly calling .filter and passing the desired query or during initialization.

In [7]:
schemes[0]

<PathScheme>: 
	categories: [chn, view, alg, scene, kind, transforms, ext]
	root: /mnt/t/DataSets/depth/Middlebury/2014_small

In [8]:
schemes[1]

<PathScheme>: 
	categories: [ext, transforms, view, alg, scene, kind, transformed, recode]
	root: /mnt/t/DataSets/evals/Middlebury

In [9]:
dc = DataCollection(*schemes,             # source
                    query="alg!='nocc'", drop=['dataset'])   # content

dc = dc.filter(query="kind!='calib'")
dc = dc.filter(query="ext!='pfm'")

In [10]:
dc

📚 Collection '<bound method PathScheme.name of <PathScheme>: 
	categories: [transforms, ext, chn, view, alg, scene, kind]
	root: /mnt/t/DataSets/depth/Middlebury/2014_small>' [544]
➿ Altered: kind!='calib' ▷ ext!='pfm'
🔖 Index[scene,kind,view,alg,chn,ext,ver,cfg] ⮕ 📒 Data[path,transforms,transformed,recode,read_trans]
 0:scene   : Adirondack, ArtL, Pipes, Playtable, Motorcycle… [12]
 1:kind    : image, rgns, disp
 2:view    : L, R
 3:alg     : c2g, ds, cam, GWC, SENSE… [18]
 4:chn     : gray, nan, RGB
 5:ext     : png, nan
 6:ver     : nan, 0.0
 7:cfg     : nan, 9411, dc4e, 3cfc, 7444… [40]

The table is accessible as a public attribute `DataCollection.db` .   
The few rows the table below show its structure: 8 levels of index and 5 data columns:  

(Tables visualization can be improved by patching their representation `df=p_*df` or printing `p_|df`)

In [11]:
dc.db[::9]

path      transforms  \
scene      kind  view alg   chn  ext ver cfg                                    
Adirondack image L    c2g   gray png NaN NaN   🖿…im0_gray.png             NaN   
ArtL       rgns  L    ds    NaN  png NaN NaN   🖿…ask0nocc.png  regions(0,1...   
Playtable  image L    c2g   gray png NaN NaN   🖿…im0_gray.png             NaN   
Motorcycle image L    cam   RGB  png NaN NaN   🖿…ycle/im0.png             NaN   
Jadeplant  image R    c2g   gray png NaN NaN   🖿…im1_gray.png             NaN   
...                                                       ...             ...   
Recycle    disp  L    LEA   NaN  NaN 0.0 7cb4  🖿…0_7cb4_L.tif         squeeze   
Playroom   disp  L    PSM   NaN  NaN 0.0 1cd0  🖿…0_1cd0_L.tif  squeeze, no...   
                      SENSE NaN  NaN 0.0 2c9a  🖿…0_2c9a_L.tif  squeeze, no...   
                      PDS   NaN  NaN 0.0 27d8  🖿…0_27d8_L.tif         squeeze   
                      SENSE NaN  NaN 0.0 4dbb  🖿…0_4dbb_L.tif  squeeze, no...   

                                                  transformed recode  \
scene      kind  view alg   chn  ext ver cfg                           
Adirondack image L    c2g   gray png NaN NaN              NaN    NaN   
ArtL       rgns  L    ds    NaN  png NaN NaN              NaN    NaN   
Playtable  image L    c2g   gray png NaN NaN              NaN    NaN   
Motorcycle image L    cam   RGB  png NaN NaN              NaN    NaN   
Jadeplant  image R    c2g   gray png NaN NaN              NaN    NaN   
...                                                       ...    ...   
Recycle    disp  L    LEA   NaN  NaN 0.0 7cb4             NaN    NaN   
Playroom   disp  L    PSM   NaN  NaN 0.0 1cd0  center_crop...    NaN   
                      SENSE NaN  NaN 0.0 2c9a             NaN    NaN   
                      PDS   NaN  NaN 0.0 27d8             NaN    NaN   
                      SENSE NaN  NaN 0.0 4dbb             NaN    NaN   

                                                   read_trans  
scene      kind  view alg   chn  ext ver cfg                   
Adirondack image L    c2g   gray png NaN NaN   O|center_cr...  
ArtL       rgns  L    ds    NaN  png NaN NaN   O|center_cr...  
Playtable  image L    c2g   gray png NaN NaN   O|center_cr...  
Motorcycle image L    cam   RGB  png NaN NaN   O|center_cr...  
Jadeplant  image R    c2g   gray png NaN NaN   O|center_cr...  
...                                                       ...  
Recycle    disp  L    LEA   NaN  NaN 0.0 7cb4  O|center_cr...  
Playroom   disp  L    PSM   NaN  NaN 0.0 1cd0  O|center_cr...  
                      SENSE NaN  NaN 0.0 2c9a  O|center_cr...  
                      PDS   NaN  NaN 0.0 27d8  O|center_cr...  
                      SENSE NaN  NaN 0.0 4dbb  O|center_cr...  

[61 rows x 5 columns]

# Working with Data
## Normalization and Transformations 
There are two major and rather standard challenges when working with large datasets:
 1. The volume of its entire data may be too large to load into the memory
 2. The raw data is not ready for homogeneous processing pipeline
 
Second point referes to misalignment in characteristics and representation of the data,  
preventing uniform processing of different data items of same kind, for multiple possible reasons:
 - images can be cropped differently, or packed differently in multi-dimentional arrays
 - data can be measured in different units, or bits representations

Even worse, those alterations and variations are often implicit, supposedely _known_ or written _somewehere_!

This framewrork suggests a structural approach this problem:
 - Make implicit explicit - dataset must be self-contained and describe all the "alterations"
 - Consider data normalization (or alignment) as a standard stage of the processing pipeline

Then those (and other) challenges can be approached using _transformations_.  
A transformation is a general operation on _data columns_:   

    `{x, y ... z} -> T -> {u, v ... w}`

That is new data lebels are created from existing (or overwrite them).  

## Complete Dataset Scheme
Scheme of the `Middlebury` below describes several aspects of the dataset:
 - `pattern` defines file-system structure and associated labeles extraction rules
 - `mappings`   renames values of _some_ of the _alg_ labels to follow convention
 - explicitely adds lablels not denied otherwise:
     - this dataset _happens_ to include only `disp`arity `left` images!
     - this critical information for processing usually appears as dataset-dependent code inserts
 - explicit _conditional_ instructions defining data-transfomations:
     - disparities produced by _some_ algorithms are scaled 
     - outputs of _some other_ algorithms are cropped from the original in certain way     
     - _conditional labeling supports two syntaxes, allowing to group by condition or category_

This particular scheme introduces two transforms-related categories: `transforms` and `transfomed` .  
Those are default names expected by the `DataCollection` class but not mandatory.
This default convention expects 
  - `transforms` to define direct instructions normalizing the data (reversing the alteration)
  - `transormed` is used when inverse transform is not available, but it _could_ be applied to align the others

%load /mnt/t/DataSets/depth/Middlebury/2014_small/info.yml

```yml

description: Subset of the Middlebury Stereo Dataset 
dataset: Middlebury 2014
domain: stereo

scheme:
    pattern: '{scene}/(?P<kind>[^\d\\/]+)(?P<view>\d)(?P<alg>[^_]*)_?(?P<chn>[^_]*)\.(?P<ext>pfm|png)'
    mappings:
        view: {"0": L, "1": R}
        kind: {im: image, mask: rgns}
        alg:  {nocc: occl}  # occlusion regions
        

    if alg == 'occl':     # `occl` is a multi-level regions map
        rgn_codes: {invl: 0, occl: 128, nocc: 255}  # with codes

    # assign `alg` and `chn` for imaging data
    if alg == '' and chn == '':
        chn: RGB
        alg: cam

    if alg == '' and chn == 'gray':
        alg: c2g             # grey images were produced by rgb2grey 

    if chn == '':  # preferable to have NaN than empty string 
        chn: null
      
    transforms if alg == 'GT':
        recode:
            from_to: [.inf, .nan]     # recode .inf -> .nan

```

%load /mnt/t/DataSets/evals/Middlebury/scheme.yml

```yml

pattern: '{scene}/(?P<alg>[-\w]+)\.(?P<ext>tif|pfm|png)'

# assign labels below unconditionally to all the items
kind: disp
view: L

transforms:                             # some images could have been in tensor dimensions
    squeeze: {}                         # apply the squeeze on all the images

# assign labels below if condition holds:
if alg in {'PSM', 'SENSE'}:         # query define when the labels below to be assigned
    transforms:                        # Label's category, followed by it's value:
        norm:                           # data undergone `norm` transformation
            inverse: True
            scale: 256                  # file_data = (data - offset) * scale

# labeling condition may be also defined as a part of the label:
transformed if alg in {'PSM'}:       # transform has been applied to PSM
    center_crop:                        # name of the transform
        width: 1200                     # named arguments of the transform
        height: 816

if alg == 'HSM':                        # this algorithm uses
    recode: [.inf, .nan]                # infinity to encode invalid pixels

transforms if alg == 'HSM':
    recode:
        from_to: [.inf, .nan]

```

```yml

pattern: '(?P<dataset>[^/_]+)/(?P<scene>\w+)/(?P<kind>[^_/]+)_(?P<alg>\w+)_(?P<ver>[^_/]+)_(?P<cfg>[^_/]+)_(?P<view>[LR]+)\.tif'

```

## Multi-Level API 
Pandas provides very powerfull toolset for re-arranging, subsetting and processing tables.  
That was a motivation to implement the `DataCollection.db` table as `pandas.DataFrame`, and to expose it to the user.  
The `DataCollection` itself provides quite thin API for the most common use-cases, so that  
user is expected to master `pandas` and use its power in the multitude of possible scenarious.

### Iterations
The most important and common use case is to iterate over certain groups of data items and apply certain processing.  
`DataCollection.iter` is a very powerfull and versatyle function, allowing to fully define the structure of such groups.  

Example below creates an iterator over subsets _grouped by scenes_ indexed by specified categories `['kind', 'alg', 'side']` .   
By default it also applies standard set of __transformations__ and __drops__ itermediate data stages:

    [path] -> read -> [data] -> transforms -> [data] -> tranformed -> [data]

In [12]:
scene, grp = g = next(dc.iter('scene', ['kind', 'alg', 'view']))
pd.set_option('max_colwidth', 10)
print(f"       Scene [{scene}]", grp, sep='\n')

       Scene [Adirondack]
                            path transforms transformed     recode read_trans
kind  alg        view                                                        
disp  ActiveS... L     🖿…e_L.tif    squeeze        NaN         NaN  O|cent...
                 L     🖿…f_L.tif    squeeze        NaN         NaN  O|cent...
                 L     🖿…4_L.tif    squeeze        NaN         NaN  O|cent...
      DSM        L     🖿…d_L.tif    squeeze        NaN         NaN  O|cent...
                 L     🖿…7_L.tif    squeeze        NaN         NaN  O|cent...
                 L     🖿…d_L.tif    squeeze        NaN         NaN  O|cent...
      GWC        L     🖿…1_L.tif    squeeze        NaN         NaN  O|cent...
                 L     🖿…c_L.tif    squeeze        NaN         NaN  O|cent...
                 L     🖿…a_L.tif    squeeze        NaN         NaN  O|cent...
      GWC_f      L     🖿…a_L.tif    squeeze        NaN         NaN  O|cent...
      HSM        L     🖿…4_L.tif  sque

In [13]:
print(type(grp))

<class 'inu.utils.pdtools.DataTable'>


If some item of the group are not required we may filter them out:

In [14]:
filter_left = lambda g: g.xs('L', level='view', drop_level=False)
itr = dc.iter('scene',['kind', 'alg', 'view'], trans=False) / filter_left
pd.set_option('max_colwidth', 10)
next(itr)

<Group> gid: Adirondack, grp:
                            path transforms transformed     recode read_trans
kind  alg        view                                                        
disp  ActiveS... L     🖿…e_L.tif    squeeze        NaN         NaN  O|cent...
                 L     🖿…f_L.tif    squeeze        NaN         NaN  O|cent...
                 L     🖿…4_L.tif    squeeze        NaN         NaN  O|cent...
      DSM        L     🖿…d_L.tif    squeeze        NaN         NaN  O|cent...
                 L     🖿…7_L.tif    squeeze        NaN         NaN  O|cent...
                 L     🖿…d_L.tif    squeeze        NaN         NaN  O|cent...
      GWC        L     🖿…1_L.tif    squeeze        NaN         NaN  O|cent...
                 L     🖿…c_L.tif    squeeze        NaN         NaN  O|cent...
                 L     🖿…a_L.tif    squeeze        NaN         NaN  O|cent...
      GWC_f      L     🖿…a_L.tif    squeeze        NaN         NaN  O|cent...
      HSM        L     🖿…4_L.tif  

To take full control over the transformations we may diable them in `iter` and apply __explicitly__ using chaining operator `/`:

In [15]:
itr = (
        dc.iter(group='scene',index=['kind', 'alg', 'view'], trans=False)
        / filter_left
        / (lambda g: apply_transforms(g, out='temp', keep='path'))
      )

## Accessing Data Elements
`DataFrame` provides flexible access to its columns, rows and index levels as to attributes and as dict keys:

In [16]:
itr = dc.iter(group='scene',index=['kind', 'alg', 'view'], trans=True, data='data')
scene, grp = next(itr)

In [17]:
grp.iloc[[0]]  # show slice of rows with indices in [0],  iloc[0] - would be the row itself

,,,data
kind,alg,view,
disp,ActiveStereo,L,[816×1...


In [18]:
grp.data[('disp', 'ActiveStereo')]

,data (Series)
view,
L,[816×1...
L,[816×1...
L,[816×1...


In [19]:
grp.data.disp.ActiveStereo.L

,data (Series)
view,
L,[816×1...
L,[816×1...
L,[816×1...


In [20]:
grp['data'].disp['ActiveStereo'].L

,data (Series)
view,
L,[816×1...
L,[816×1...
L,[816×1...


In [21]:
grp.data[:10]

data (Series)
kind alg        view              
disp ActiveS... L     [816×1...   
                L     [816×1...   
                L     [816×1...   
     DSM        L     [816×1...   
                L     [816×1...   
                L     [816×1...   
     GWC        L     [816×1...   
                L     [816×1...   
                L     [816×1...   
     GWC_f      L     [816×1...

That allows expressive and concise syntax and integration with other functions: 

In [ ]:
for i, (scene, grp) in enumerate(itr):    
    if i >= 2: break;         
    imgrid(grp.data.image, clim=[0, 1], mosaic='titles')
    imgrid(grp.data.disp, clim=[0, 1], mosaic='titles')
    imgrid(grp.data.rgns, clim=[0, 1], mosaic='titles')

# Building Workflows
## Assumptions

There are some general assumptions which once satisfied simplifies greately code and workflows:
 - Invalid data elements `invl` are represented by same value (`np.nan`) 
    - `np.nan` is a preferable choice for its wide support by `numpy` and `pandas`
    - necessary exceptions may include `integer` data types, where special traetment is required
      (pandas started to support `np.nan` even in integer arrays!)
    - invalids are by default excluded from standard calculations (metrics, etc)    
 - Data arrays are spatially aligned - same mapping between their indices and `x,y`

## Stable Labeling
The datasets framework `inu.ds` is quite generic and makes very few assumptions.  
However smooth work with data requires simplicity and stability not only of its _explicit API_ ,  
but also of the _implicit_ one - the data structure itself, defined mainly the _data labels_.  

Therefore its important to follow two general guidelines:
1. Turn all _implicit_ knowledge about the dataset into _explicit_ through `schemes`.
2. Make the _labeling_ and codes of categories _stable_.

Labeling conventions are expected to be specific to a particular application domain.  

## Typical Workflow Components
Various data processing scenarios, especially in a specific domain, share stages and concepts,  
which, once formalized, streamline thinking and coding.

Constructs presented below are typical for the stereo domain, but also way beyond.

### Regions Maps
Elements in data (like pixels in image) may be associated with different semantic classes.  
This information is defined using _region maps_ .    
In general form a _region map_ is associated with every _region kind_ (semantic class):  `{region_kind: region_map}`  

If classes are not itersected a _single region map_ may be used with regions differentiated by _region codes_ .      
That also asumes that association to the classes is _binary_, that is a pixel is either belongs to a class or not.  

Sometimes semantic 'painting' is multi-level, and the region maps are not represented by binary masks.  
Multi-level classes could be collapsed into binary for simplification.

   > For region-based metrics read on [...Evaluation of Stereo Algorithms](https://vision.middlebury.edu/stereo/taxonomy-IJCV.pdf)

### Some common semantic codes:

| Code   | bin | Semantics   | Comments  
|--------|:---:|-------------|---------------
| `invl` |  Y  | no GT       | Semantic map version of invalids
| `occl` |  Y  | occlusion   | from the GT dataset or calculated
| `edge` |  N  | object edge | near the discontinuity in the depth of objects
| `tilt` |  N  | tilted      | Areas with normals tilted from the cameras axis 
| `txtr` |  N  | texture     | Spacial variability in the image (depends on threshold)      
| `cnfd` |  N  | confidence  | Some algorithms produce measure of their confidence
| `devt` |  N  | deviation   | Based on observed deviation from the GT

### Composed Regions
Independent region classes may combined into new categories: `bad_edge` = `err` & `dscn` .  
In general that can be described as operations on categorical sets: 
$$C_{bad\ edge} = C_{err} \cap C_{dscn}$$

## Uniform Metrics
In general terms a _metric_ as an arbitrary function corresponding a collection of elements to a single number. 
Most of the metrics are statistical function $F_S$ processing their input set $V$ _uniformly_,
that is __not sensitive to the order or spacial location__ of those values $M = F_S(V)$

This set $V$ of values generally is a product of chain of operatioins, for _example_  
starting from the input data, like 
   - images $I_{L,R}$, 
   - disparities $D_{L,R}$ 
   - ground truth disparity $G_{L,R}$        

then passing through some categorizations:
   $$\begin{aligned}
   C_{err} &= |G-D| > T_{err} \\
   C_{dscn} &= \mathrm{canny}(I)
   \end{aligned}$$
then through categories based filtering:
    $$ V = D \in (C_{err} \cap C_{dscn}) $$
to the final statiscical processing $F_S(V)$.

That allows to define a generic `uniform_metrics` function 
 - applying arbitrary _measurements_ 
 - _differentially_ by specific _regions_ in the data 
 
`M = uniform_metrics(input_set, measures, regions)`